<a href="https://colab.research.google.com/github/carlocamilloni/Structural-Bioinformatics/blob/main/Notebooks/t07_QM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Quantum chemistry with ORCA

In the previous practicals the atoms were classical particles interacting through a **force field**: a set of simple functions (bonds, angles, torsions, point charges, Lennard-Jones) whose parameters were fitted, to a large extent, to quantum-mechanical (QM) calculations. Here we go one level down and treat the **electrons** explicitly: we solve (approximately) the electronic Schrödinger equation for a fixed position of the nuclei and get the energy, the electron density, the molecular orbitals and everything that follows from them: geometries, vibrations, interaction energies, colours.

We use [ORCA](https://www.faccts.de/orca/), one of the most widely used quantum-chemistry programs, free for academic use.

The reference lecture is:

*   https://github.com/carlocamilloni/Structural-Bioinformatics/blob/main/Notes/08_QM_MM_more.pdf

The practical has a training part, that **everybody** runs, and a report with **two different exercises**: **GROUP 1** studies the **noncovalent interactions** that hold biomolecules together (hydrogen bonds and stacking of nucleobases, aromatic side chains), and **GROUP 2** the **colour** of biological chromophores (the GFP chromophore, retinal, flavins, aromatic amino acids) with time-dependent DFT. Parts 4 and 5 of the training are the starting points of the two exercises.

**Training**

1.  **One electron: the hydrogen atom.** Basis sets, Hartree–Fock vs DFT, the electron density and the Bohr radius.
2.  **More electrons: the carbon atom.** Spin multiplicity, Hund's rule, orbital energies.
3.  **The chemical bond: H$_2$.** Bonding and antibonding orbitals, where the bond electrons are, and what goes wrong when a bond is broken.
4.  **A real molecule: water.** Geometry optimisation, vibrational frequencies and the IR spectrum, dipole moment and atomic charges, and how the results depend on the method; then two water molecules and the energy of a hydrogen bond.
5.  **Light: excited states.** The UV absorption of formaldehyde and butadiene with TD-DFT.

**Units.** Quantum chemistry uses *atomic units*: energies are in Hartree (1 E$_h$ = 27.211 eV = 627.51 kcal/mol = 2625.5 kJ/mol) and lengths in bohr (1 bohr = 0.529 Å). ORCA reads coordinates in Å.

## Preliminary setup

All the calculations of the training are small and run on the CPU, so you don't need a GPU. Colab gives you 2 CPU cores; we run ORCA on one core (simpler and robust), the timings reported below refer to this setting.

### Installing ORCA

[ORCA](https://www.faccts.de/orca/) is free for academic use, but **it cannot be redistributed**: everybody has to download their own copy. You do this only once:

1. Register on the [ORCA forum](https://orcaforum.kofo.mpg.de) with your university e-mail and accept the licence (the account is activated in a few minutes).
2. In `Downloads`, open the folder of the most recent ORCA 6 version and download the **Linux, x86-64, shared-version** archive, a file with a name like `orca_6_1_1_linux_x86-64_shared_openmpi418_avx2.tar.xz` (about 1 GB). Don't take the macOS or Windows versions: Colab runs Linux.
3. Upload the archive to your Google Drive, in the folder `MyDrive/Structural_Bioinformatics/ORCA` (create it if needed). Don't unpack it.

The licence is personal: **don't share your copy** with other people (and don't put it in a shared folder).

The next cell mounts your Google Drive, finds the archive and unpacks it in the virtual machine (1–2 minutes). Colab forgets everything at the end of the session: **run it again every time you start a new session**. If you run the notebook on your own Linux computer and ORCA is already installed, set the variable `ORCA_DIR` to the ORCA folder (the one containing the program `orca`) and the cell only checks the installation.

In [ ]:
import os, glob, shutil, subprocess

ORCA_DIR = os.environ.get('ORCA_DIR', '')      # on your own computer: the folder containing the program orca
DRIVE_FOLDER = '/content/drive/MyDrive/Structural_Bioinformatics'

if not (ORCA_DIR and os.path.isfile(os.path.join(ORCA_DIR, 'orca'))):
    from google.colab import drive
    drive.mount('/content/drive')
    archives = sorted(glob.glob(os.path.join(DRIVE_FOLDER, 'ORCA', 'orca_6*linux_x86-64*.tar.*')))
    if not archives:
        raise FileNotFoundError('No ORCA archive found in ' + os.path.join(DRIVE_FOLDER, 'ORCA') +
                                ': follow the instructions above to download it.')
    archive = archives[-1]                        # if there is more than one, take the most recent version
    ORCA_DIR = os.path.join('/content', os.path.basename(archive).split('.tar')[0])
    if not os.path.isfile(os.path.join(ORCA_DIR, 'orca')):
        print('unpacking', os.path.basename(archive), '...')
        subprocess.run(['tar', '-xf', archive, '-C', '/content'], check=True)
    # some archives contain an additional folder level
    found = glob.glob(os.path.join('/content', 'orca_6*', '**', 'orca'), recursive=True)
    ORCA_DIR = os.path.dirname(sorted(found)[-1])

# ORCA must be called with its full path; the shared version also needs its libraries
os.environ['PATH'] = ORCA_DIR + ':' + os.environ['PATH']
os.environ['LD_LIBRARY_PATH'] = ORCA_DIR + ':' + os.environ.get('LD_LIBRARY_PATH', '')
os.environ['OMP_NUM_THREADS'] = '1'
ORCA = os.path.join(ORCA_DIR, 'orca')
version = subprocess.run([ORCA, '--version'], capture_output=True, text=True).stdout.strip()
print('ORCA found in', ORCA_DIR)
print(version if version else 'ORCA is installed')

We also install `py3Dmol` to look at molecules, orbitals and densities directly in the notebook.

In [ ]:
%pip install -q py3Dmol

We work in a folder on your Google Drive (`MyDrive/Structural_Bioinformatics/T07`): the calculations of the training are reused in the report, and the files survive a disconnection of Colab. On your own computer the folder `T07` is made next to the notebook. The cell can be run more than once.

In [ ]:
if 'BASE_DIR' not in globals():          # remember where we started: running this cell again does not go one folder deeper
    BASE_DIR = os.getcwd()
if os.path.isdir('/content/drive/MyDrive'):
    local_path = os.path.join(DRIVE_FOLDER, 'T07')
else:
    local_path = os.path.join(BASE_DIR, 'T07')
os.makedirs(local_path, exist_ok=True)
%cd $local_path

# the input structures are in the Data/qm folder of the course repository:
# if this notebook is run from a copy of the repository we use it, otherwise we download the repository
if os.path.isdir(os.path.join(BASE_DIR, '..', 'Data', 'qm', 's22')):
    REPO_DIR = os.path.abspath(os.path.join(BASE_DIR, '..'))
else:
    REPO_DIR = '/content/sb' if os.path.isdir('/content') else os.path.join(BASE_DIR, 'sb')
    if os.path.isdir(REPO_DIR):
        !cd "$REPO_DIR" && git pull -q
    else:
        !git clone -q --depth 1 https://github.com/carlocamilloni/Structural-Bioinformatics.git "$REPO_DIR"
QM_DIR = os.path.join(REPO_DIR, 'Data', 'qm')
print('input files from', QM_DIR)

### A few helper functions

ORCA reads a text **input file** (`name.inp`) and writes a long text **output file** (`name.out`), plus a few other files that start with the same name. The input always has the same structure:

```
! method basis-set other-keywords        <- the "simple input" line(s)
%block                                   <- optional blocks with more detailed settings
  keyword value
end
* xyz charge multiplicity                <- the molecule: charge, spin multiplicity and Cartesian coordinates in Å
H  0.0  0.0  0.0
*
```

The functions below write the input, run ORCA, and extract the numbers we need from the output. Read them: they are short and there is nothing magic in them. Every calculation is run in its own folder, so that the files of different calculations don't mix. If a calculation has already been run (its output exists and ends normally), it is **not run again**: this saves a lot of time when you rerun the notebook. Use `rerun=True` to force a new run.

In [ ]:
import re, time
import numpy as np
import matplotlib.pyplot as plt
import py3Dmol

HARTREE2EV = 27.211386        # 1 Eh in eV
HARTREE2KCAL = 627.5095       # 1 Eh in kcal/mol
BOHR2ANG = 0.529177           # 1 bohr in Å

def run_orca(name, inp, folder='.', rerun=False):
    # write folder/name/name.inp, run ORCA in folder/name and return the path of the output file
    workdir = os.path.join(folder, name)
    os.makedirs(workdir, exist_ok=True)
    out = os.path.join(workdir, name + '.out')
    if not rerun and os.path.isfile(out) and 'ORCA TERMINATED NORMALLY' in open(out).read():
        print(f'{name}: already done, reading {out}')
        return out
    with open(os.path.join(workdir, name + '.inp'), 'w') as f:
        f.write(inp.strip() + '\n')
    t0 = time.time()
    with open(out, 'w') as f:
        subprocess.run([ORCA, name + '.inp'], cwd=workdir, stdout=f, stderr=subprocess.STDOUT)
    text = open(out).read()
    if 'ORCA TERMINATED NORMALLY' not in text:
        print(text[-3000:])
        raise RuntimeError(f'ORCA did not terminate normally: look at the end of {out}')
    print(f'{name}: done in {time.time() - t0:.1f} s')
    return out

def xyz_block(xyzfile, ghost=()):
    # the coordinate lines of an xyz file; the atoms whose index is in ghost become ghost atoms (see Part 4)
    lines = open(xyzfile).read().splitlines()
    natoms = int(lines[0])
    atoms = []
    for i, l in enumerate(lines[2:2 + natoms]):
        el, x, y, z = l.split()[:4]
        atoms.append(f'{el}{":" if i in ghost else " "} {x:>12} {y:>12} {z:>12}')
    return '\n'.join(atoms)

def final_energy(out):
    # the last 'FINAL SINGLE POINT ENERGY' of an output (in Eh)
    return float(re.findall(r'FINAL SINGLE POINT ENERGY\s+(-?\d+\.\d+)', open(out).read())[-1])

def wall_time(out):
    # total run time of a calculation, in seconds
    d, h, m, s, ms = re.findall(r'TOTAL RUN TIME:\s+(\d+) days (\d+) hours (\d+) minutes (\d+) seconds (\d+) msec', open(out).read())[-1]
    return 86400*int(d) + 3600*int(h) + 60*int(m) + int(s) + int(ms)/1000

def s2(out):
    # expectation value of S^2 (only for UHF/UKS calculations)
    return float(re.findall(r'Expectation value of <S\*\*2>\s*:\s*(\d+\.\d+)', open(out).read())[-1])

def dipole(out):
    # magnitude of the dipole moment, in Debye
    return float(re.findall(r'Magnitude \(Debye\)\s*:\s*(\d+\.\d+)', open(out).read())[-1])

def orbital_energies(out):
    # the last table of orbital energies: list of (occupation, energy in eV); for UHF the spin-up orbitals
    text = open(out).read().split('ORBITAL ENERGIES')[-1]
    rows = []
    for l in text.splitlines()[4:]:
        f = l.split()
        if len(f) != 4:
            if rows: break
            continue
        rows.append((float(f[1]), float(f[3])))
    return rows

def charges(out, kind='MULLIKEN'):
    # atomic charges: kind is MULLIKEN, LOEWDIN, HIRSHFELD or CHELPG
    text = open(out).read()
    if kind == 'MULLIKEN':
        block = text.split('MULLIKEN ATOMIC CHARGES')[-1].split('Sum of atomic charges')[0]
    elif kind == 'LOEWDIN':
        block = text.split('LOEWDIN ATOMIC CHARGES')[-1].split('LOEWDIN REDUCED')[0]
    elif kind == 'HIRSHFELD':
        block = text.split('HIRSHFELD ANALYSIS')[-1].split('TOTAL')[1]
    elif kind == 'CHELPG':
        block = text.split('CHELPG Charges')[-1].split('Total charge')[0]
    return [(el, float(q)) for el, q in re.findall(r'^\s*\d+\s+([A-Z][a-z]?)\s*:?\s+(-?\d+\.\d+)', block, re.M)]

def read_xyz(xyzfile):
    lines = open(xyzfile).read().splitlines()
    n = int(lines[0])
    el = [l.split()[0] for l in lines[2:2+n]]
    xyz = np.array([[float(v) for v in l.split()[1:4]] for l in lines[2:2+n]])
    return el, xyz

def read_cube(cubefile):
    # Gaussian cube file -> (origin, axes, values); origin and axes in bohr
    with open(cubefile) as f:
        f.readline(); f.readline()
        n, *origin = f.readline().split()
        natoms = abs(int(n))
        axes, shape = [], []
        for _ in range(3):
            npt, *v = f.readline().split()
            shape.append(int(npt)); axes.append([float(x) for x in v])
        for _ in range(natoms):
            f.readline()
        data = np.array(f.read().split(), dtype=float)
    return np.array(origin, dtype=float), np.array(axes), data[-np.prod(shape):].reshape(shape)

def show_molecule(xyzfile, width=400, height=300):
    view = py3Dmol.view(width=width, height=height)
    view.addModel(open(xyzfile).read(), 'xyz')
    view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'scale': 0.25}})
    view.zoomTo()
    return view.show()

def show_cube(cubefile, isovalue=0.05, signed=False, xyzfile=None, width=400, height=300):
    # isosurface of a cube file; signed=True for orbitals (positive lobes red, negative lobes blue)
    view = py3Dmol.view(width=width, height=height)
    data = open(cubefile).read()
    view.addModel(data, 'cube')
    view.setStyle({'stick': {'radius': 0.12}, 'sphere': {'scale': 0.2}})
    view.addVolumetricData(data, 'cube', {'isoval': isovalue, 'color': 'red', 'opacity': 0.8})
    if signed:
        view.addVolumetricData(data, 'cube', {'isoval': -isovalue, 'color': 'blue', 'opacity': 0.8})
    view.zoomTo()
    return view.show()

def plots_block(cubes, npoints=60):
    # an ORCA %plots block that writes cube files; cubes is a list of strings like
    # 'ElDens("x.eldens.cube")' or 'MO("x.mo4.cube",4,0)' (orbitals are numbered from 0)
    return '%plots\n  dim1 {0}\n  dim2 {0}\n  dim3 {0}\n  Format Gaussian_Cube\n'.format(npoints) + \
           ''.join(f'  {c};\n' for c in cubes) + 'end\n'

## Part 1: one electron, the hydrogen atom

The hydrogen atom is the only system for which we know the exact solution: the ground-state energy is exactly $-0.5$ E$_h$ (= −13.6 eV, minus the ionisation energy), and the wavefunction is $\psi_{1s}(r) = e^{-r/a_0}/\sqrt{\pi a_0^3}$, where $a_0$ = 0.529 Å is the Bohr radius.

A quantum-chemistry program writes the orbitals as a combination of a finite number of functions centred on the atoms, the **basis set**. Bigger basis sets are more flexible and more accurate, but more expensive. We use the *def2* family: def2-SVP (split valence + polarisation, small), def2-TZVP (triple zeta, medium), def2-QZVPP (quadruple zeta, large).

The first input: one H atom, charge 0, multiplicity 2 (one unpaired electron: $2S+1 = 2$), Hartree–Fock (HF) with the unrestricted formalism (UHF, spin-up and spin-down electrons have different orbitals) and the def2-SVP basis set:

In [ ]:
os.makedirs('H', exist_ok=True)
inp = '''
! UHF def2-SVP TightSCF
* xyz 0 2
H  0.0 0.0 0.0
*
'''
out = run_orca('H_HF_def2-SVP', inp, folder='H')

Look at the output file (`T07/H/H_HF_def2-SVP/H_HF_def2-SVP.out`, double-click on it in the file browser on the left). It is long, but it is organised in sections. Find:

*   `BASIS SET INFORMATION`: how many basis functions are used;
*   `SCF ITERATIONS`: the self-consistent field procedure. Starting from a guess, the orbitals are improved iteratively until the energy no longer changes (`Delta-E`);
*   `TOTAL SCF ENERGY` and, at the end, `FINAL SINGLE POINT ENERGY`;
*   `ORBITAL ENERGIES`: the energy of each orbital and how many electrons occupy it.

In [ ]:
print(open(out).read().split('SCF ITERATIONS')[1][:1500])
print('Final energy:', final_energy(out), 'Eh')

Now we repeat the calculation with bigger basis sets and with two density functionals: **PBE** (a GGA functional) and **B3LYP** (a hybrid functional, containing 20% of HF exchange). All the calculations together take less than a minute.

In [ ]:
results = []
for method in ['UHF', 'UKS PBE', 'UKS B3LYP']:
    for basis in ['def2-SVP', 'def2-TZVP', 'def2-QZVPP']:
        name = f"H_{method.split()[-1]}_{basis}"
        out = run_orca(name, f'! {method} {basis} TightSCF\n* xyz 0 2\nH 0 0 0\n*', folder='H')
        results.append((method.split()[-1], basis, final_energy(out)))

print(f"{'method':8s} {'basis':12s} {'E (Eh)':>12s} {'E - exact (kcal/mol)':>22s}")
for method, basis, e in results:
    print(f'{method:8s} {basis:12s} {e:12.6f} {(e + 0.5) * HARTREE2KCAL:22.2f}')

With the largest basis set HF is (almost) exact, while the two functionals are not, and the error does **not** go to zero when the basis set is improved. For one electron the HF equations are exact (there is no other electron to correlate with) while in DFT the electron interacts with its own density: this is the *self-interaction error* of approximate functionals. Keep in mind the order of magnitude of these errors (kcal/mol): a hydrogen bond is worth a few kcal/mol.

### The electron density

The electron density $\rho(\mathbf{r})$ is the probability per unit volume of finding an electron at $\mathbf{r}$ (multiplied by the number of electrons). It is what X-ray crystallography measures. ORCA writes it on a 3D grid in a *cube* file with the `%plots` block; here we use a fine grid (80 points per side) around the atom:

In [ ]:
inp = '! UHF def2-QZVPP TightSCF\n' + plots_block(['ElDens("H.eldens.cube")'], npoints=80) + '* xyz 0 2\nH 0 0 0\n*'
out = run_orca('H_density', inp, folder='H')
show_cube('H/H_density/H.eldens.cube', isovalue=0.01)

Change the isovalue (0.2, 0.05, 0.01, 0.001 electrons/bohr$^3$): the density decays exponentially but never becomes exactly zero. You can also open the cube file with VMD (`File → New Molecule`, then `Graphical Representations → Isosurface`) or ChimeraX (`open H.eldens.cube`).

The cube file is just a 3D array of numbers. We can read it and compute $\rho(r)$ as a function of the distance from the nucleus, and compare it with the exact $\rho(r) = |\psi_{1s}|^2 = e^{-2r/a_0}/(\pi a_0^3)$. The **radial distribution** $P(r) = 4\pi r^2 \rho(r)$ is the probability of finding the electron at distance $r$, in a shell of thickness d$r$.

In [ ]:
origin, axes, rho = read_cube('H/H_density/H.eldens.cube')
# coordinates (in bohr) of the grid points
idx = np.indices(rho.shape).reshape(3, -1).T
points = origin + idx @ axes
r = np.linalg.norm(points, axis=1)          # distance from the nucleus (at the origin), in bohr
rho = rho.ravel()
print('number of electrons in the box:', rho.sum() * abs(np.linalg.det(axes)))

order = np.argsort(r)
r_ex = np.linspace(0, r.max(), 300)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].semilogy(r[order] * BOHR2ANG, rho[order], '.', ms=2, label='ORCA, UHF/def2-QZVPP')
ax[0].semilogy(r_ex * BOHR2ANG, np.exp(-2 * r_ex) / np.pi, 'k-', label='exact')
ax[0].set_xlabel('r (Å)'); ax[0].set_ylabel(r'$\rho$ (bohr$^{-3}$)'); ax[0].legend()
ax[1].plot(r[order] * BOHR2ANG, 4 * np.pi * r[order]**2 * rho[order], '.', ms=2)
ax[1].plot(r_ex * BOHR2ANG, 4 * r_ex**2 * np.exp(-2 * r_ex), 'k-')
ax[1].axvline(BOHR2ANG, ls='--', c='grey')
ax[1].set_xlabel('r (Å)'); ax[1].set_ylabel(r'$4\pi r^2 \rho$ (bohr$^{-1}$)')
plt.tight_layout(); plt.show()

The radial distribution has its maximum at the Bohr radius (dashed line). At large $r$ the calculated density deviates from the exact one: the basis functions are Gaussians ($e^{-\alpha r^2}$) that decay too quickly, while the exact density is an exponential.

## Part 2: more electrons, the carbon atom

Carbon has 6 electrons, $1s^2 2s^2 2p^2$. Unlike the pseudopotential codes used for materials, ORCA treats all the electrons explicitly, also the $1s$ core electrons. How should the two $2p$ electrons be arranged? With the same spin in two different $p$ orbitals (total spin $S=1$, multiplicity $2S+1=3$, a *triplet*) or paired in the same orbital (singlet)? And what if one $2s$ electron is promoted to $2p$ ($2s^1 2p^3$, four unpaired electrons, a quintet, the configuration used to explain the four bonds of carbon)?

We compute the energy of the three spin states. We use B3LYP and the def2-TZVP basis set.

In [ ]:
os.makedirs('C', exist_ok=True)
energies = {}
for mult in [1, 3, 5]:
    inp = f'! UKS B3LYP def2-TZVP TightSCF\n' + \
          (plots_block([f'SpinDens("C_{mult}.spindens.cube")']) if mult > 1 else '') + \
          f'* xyz 0 {mult}\nC 0 0 0\n*'
    out = run_orca(f'C_mult{mult}', inp, folder='C')
    energies[mult] = final_energy(out)
    print(f'multiplicity {mult}: E = {energies[mult]:.6f} Eh   <S^2> = {s2(out):.3f}')
for mult in energies:
    print(f'multiplicity {mult}: E - E(triplet) = {(energies[mult] - energies[3]) * HARTREE2EV:6.2f} eV')

In the output of each calculation, look at the `ORBITAL ENERGIES` table: there are two tables, for the spin-up (alpha) and spin-down (beta) electrons, with the occupation (OCC) of each orbital. The function `orbital_energies()` reads the spin-up one:

In [ ]:
for occ, e in orbital_energies('C/C_mult3/C_mult3.out')[:8]:
    print(f'occupation {occ:.1f}   energy {e:9.3f} eV')

The **spin density** (density of spin-up minus spin-down electrons) shows where the unpaired electrons are:

In [ ]:
show_cube('C/C_mult3/C_3.spindens.cube', isovalue=0.01)

## Part 3: the chemical bond, H$_2$

Two hydrogen atoms make a bond. In the simplest picture the two $1s$ orbitals combine into a **bonding** orbital $\sigma_g \propto 1s_A + 1s_B$ (lower energy, occupied by the two electrons with opposite spin) and an **antibonding** orbital $\sigma_u^* \propto 1s_A - 1s_B$ (higher energy, empty).

First we compute H$_2$ at the experimental bond length (0.741 Å), write the two orbitals and the density in cube files, and look at them. Orbitals are numbered from 0 in ORCA: orbital 0 is $\sigma_g$ and orbital 1 is $\sigma_u^*$. To compare the densities of H$_2$ and of two isolated H atoms we need the same grid, so here we write the `%plots` block by hand fixing the box (in bohr).

In [ ]:
os.makedirs('H2', exist_ok=True)
grid = '''%plots
  dim1 60
  dim2 60
  dim3 80
  min1 -4.0
  max1  4.0
  min2 -4.0
  max2  4.0
  min3 -5.0
  max3  5.0
  Format Gaussian_Cube
'''
geom = '* xyz {} {}\nH 0 0 -0.3705\nH 0 0 0.3705\n*'
out = run_orca('H2_orbitals', '! RHF def2-TZVP TightSCF\n' + grid +
               '  MO("H2.mo0.cube",0,0);\n  MO("H2.mo1.cube",1,0);\n  ElDens("H2.eldens.cube");\nend\n' + geom.format(0, 1), folder='H2')
# the "promolecule": two isolated H atoms at the same positions, each with its own density (one electron each)
for i, z in enumerate([-0.3705, 0.3705]):
    run_orca(f'H_atom{i}', '! UHF def2-TZVP TightSCF\n' + grid + f'  ElDens("H{i}.eldens.cube");\nend\n* xyz 0 2\nH 0 0 {z}\n*', folder='H2')
for occ, e in orbital_energies(out)[:2]:
    print(f'occupation {occ:.1f}   energy {e:8.2f} eV')

In [ ]:
show_cube('H2/H2_orbitals/H2.mo0.cube', isovalue=0.05, signed=True)
show_cube('H2/H2_orbitals/H2.mo1.cube', isovalue=0.05, signed=True)

Where do the electrons go when the bond is formed? The **density difference** $\Delta\rho = \rho_{H_2} - \rho_{H_A} - \rho_{H_B}$ shows it: positive where the bond accumulates electrons, negative where it removes them. Below we plot it on a plane containing the two nuclei.

In [ ]:
o, ax, rho_h2 = read_cube('H2/H2_orbitals/H2.eldens.cube')
_, _, rho_a = read_cube('H2/H_atom0/H0.eldens.cube')
_, _, rho_b = read_cube('H2/H_atom1/H1.eldens.cube')
drho = rho_h2 - rho_a - rho_b
mid = rho_h2.shape[1] // 2                                          # the plane y = 0
x = (o[0] + np.arange(rho_h2.shape[0]) * ax[0, 0]) * BOHR2ANG
z = (o[2] + np.arange(rho_h2.shape[2]) * ax[2, 2]) * BOHR2ANG
fig, axs = plt.subplots(1, 2, figsize=(10, 4.5))
c0 = axs[0].contourf(z, x, rho_h2[:, mid, :], levels=np.linspace(0, 0.3, 31), cmap='viridis')
axs[0].set_title(r'$\rho$(H$_2$)'); plt.colorbar(c0, ax=axs[0])
lim = np.abs(drho[:, mid, :]).max()
c1 = axs[1].contourf(z, x, drho[:, mid, :], levels=np.linspace(-lim, lim, 31), cmap='RdBu_r')
axs[1].set_title(r'$\Delta\rho = \rho$(H$_2$) $- \rho$(H) $- \rho$(H)'); plt.colorbar(c1, ax=axs[1])
for a in axs:
    a.plot([-0.3705, 0.3705], [0, 0], 'k+', ms=12); a.set_xlabel('z (Å)'); a.set_ylabel('x (Å)'); a.set_aspect('equal')
plt.tight_layout(); plt.show()

### Breaking the bond

Now we stretch the bond from 0.5 to 4 Å and compute the energy curve with three methods:

*   **RHF**: restricted Hartree–Fock, the two electrons share the same spatial orbital $\sigma_g$;
*   **UHF**: unrestricted Hartree–Fock, the spin-up and spin-down electron can have different orbitals. To let the calculation find this solution we start from the triplet and flip the spin of one electron (`FlipSpin`), a *broken-symmetry* calculation;
*   **CCSD**: coupled cluster with single and double excitations. It includes the *electron correlation* missing in HF and, for two electrons, it is exact within the basis set (it is equivalent to full configuration interaction).

The 3 × 15 calculations take about 2 minutes.

In [ ]:
distances = np.array([0.5, 0.6, 0.65, 0.7, 0.75, 0.8, 0.9, 1.0, 1.2, 1.4, 1.7, 2.0, 2.5, 3.0, 4.0])
methods = {'RHF': '! RHF def2-TZVP TightSCF',
           'UHF': '! UHF def2-TZVP TightSCF\n%scf\n  FlipSpin 1\n  FinalMs 0.0\nend',
           'CCSD': '! CCSD def2-TZVP TightSCF'}
curves = {m: [] for m in methods}
spin = []
for m, keywords in methods.items():
    for d in distances:
        mult = 3 if m == 'UHF' else 1           # FlipSpin starts from the triplet and flips one electron
        out = run_orca(f'H2_{m}_{d:.2f}', keywords + f'\n* xyz 0 {mult}\nH 0 0 0\nH 0 0 {d}\n*', folder='H2')
        curves[m].append(final_energy(out))
        if m == 'UHF':
            spin.append(s2(out))
curves = {m: np.array(e) for m, e in curves.items()}

In [ ]:
# energy of two isolated H atoms with the same basis set: twice the UHF energy of one atom (exact for HF and CCSD)
e_atom = final_energy(run_orca('H_UHF_def2-TZVP', '! UHF def2-TZVP TightSCF\n* xyz 0 2\nH 0 0 0\n*', folder='H2'))
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for m, e in curves.items():
    ax[0].plot(distances, (e - 2 * e_atom) * HARTREE2EV, 'o-', label=m)
ax[0].axhline(0, c='grey', ls='--'); ax[0].set_ylim(-5, 4)
ax[0].set_xlabel('H–H distance (Å)'); ax[0].set_ylabel('E − 2 E(H) (eV)'); ax[0].legend()
ax[1].plot(distances, spin, 'o-', c='C1'); ax[1].set_xlabel('H–H distance (Å)'); ax[1].set_ylabel(r'UHF $\langle S^2 \rangle$')
plt.tight_layout(); plt.show()
for m, e in curves.items():
    i = np.argmin(e)
    print(f'{m:5s}: minimum at {distances[i]:.2f} Å, E(min) - 2E(H) = {(e[i] - 2 * e_atom) * HARTREE2EV:6.2f} eV, '
          f'E(4 Å) - 2E(H) = {(e[-1] - 2 * e_atom) * HARTREE2EV:6.2f} eV')

The experimental bond length is 0.741 Å and the dissociation energy $D_e$ (from the bottom of the curve) is 4.75 eV (109.5 kcal/mol). Notice:

*   RHF does not go to the energy of two H atoms at large distance: forcing the two electrons in the same orbital, $\sigma_g = 1s_A + 1s_B$, means that 50% of the time both electrons are on the same atom (H$^+$ + H$^-$);
*   UHF dissociates correctly but, for this to happen, the spin-up electron goes on one atom and the spin-down on the other: the wavefunction is no longer a pure singlet ($\langle S^2 \rangle$ becomes 1 instead of 0);
*   at the minimum, both HF curves miss part of the binding: this is the **correlation energy**, recovered by CCSD.

Breaking bonds (chemical reactions, transition metals, radicals) is the hard part of quantum chemistry. For the noncovalent interactions and the excited states of closed-shell molecules that we study next, the standard methods work well.

## Part 4: a real molecule, water

### Geometry optimisation and vibrational frequencies

We start from a rough structure of water and let ORCA find the minimum of the energy (`Opt`), then compute the vibrational frequencies (`Freq`): the second derivatives of the energy with respect to the positions of the nuclei give the force constants, and from those the normal modes. We use **r$^2$SCAN-3c**, a modern "composite" method designed to be accurate and cheap: the r$^2$SCAN functional with a tailored basis set and corrections for dispersion (D4) and for the incompleteness of the basis set.

`! CHELPG` adds a second type of atomic charges, fitted to the electrostatic potential around the molecule (the same idea used to derive the charges of AMBER force fields).

In [ ]:
os.makedirs('water', exist_ok=True)
water = '''* xyz 0 1
O   0.000   0.000   0.000
H   0.000   0.800   0.600
H   0.000  -0.800   0.600
*'''
inp = '! r2SCAN-3c Opt Freq CHELPG TightSCF\n' + plots_block(['MO("water.homo.cube",4,0)', 'MO("water.lumo.cube",5,0)']) + water
out = run_orca('water_r2SCAN-3c', inp, folder='water')
show_molecule('water/water_r2SCAN-3c/water_r2SCAN-3c.xyz')

In [ ]:
def geometry(xyzfile):
    # O-H distances and H-O-H angle of a water molecule (atoms in the order O, H, H)
    el, x = read_xyz(xyzfile)
    d1, d2 = x[1] - x[0], x[2] - x[0]
    angle = np.degrees(np.arccos(d1 @ d2 / np.linalg.norm(d1) / np.linalg.norm(d2)))
    return np.linalg.norm(d1), np.linalg.norm(d2), angle

def frequencies(out):
    # vibrational frequencies (cm-1) and IR intensities (km/mol) from the IR SPECTRUM table
    text = open(out).read().split('IR SPECTRUM')[-1].split('\n\n\n')[0]
    rows = re.findall(r'^\s*(\d+):\s+(-?\d+\.\d+)\s+\S+\s+(\d+\.\d+)', text, re.M)
    return np.array([[float(f), float(i)] for _, f, i in rows])

r1, r2, angle = geometry('water/water_r2SCAN-3c/water_r2SCAN-3c.xyz')
print(f'O-H = {r1:.4f} Å   H-O-H = {angle:.2f} degrees      (experiment: 0.9572 Å, 104.52 degrees)')
print(f'dipole = {dipole(out):.2f} D                        (experiment: 1.85 D)')
print('frequencies (cm-1) and IR intensities (km/mol):')
print(frequencies(out))

The three frequencies correspond to the bending, the symmetric stretching and the antisymmetric stretching of water. The experimental *fundamental* frequencies in the gas phase are 1595, 3657 and 3756 cm$^{-1}$; the *harmonic* ones (what we compute, the curvature at the minimum, without anharmonicity) are 1649, 3832 and 3943 cm$^{-1}$.

The normal modes are in the output (section `NORMAL MODES`): ORCA writes a movie of each one with `orca_pltvib`, which we can play with py3Dmol (modes 0–5 are translations and rotations, 6–8 are the vibrations).

In [ ]:
def show_mode(out, mode, width=400, height=300):
    folder, name = os.path.split(out)
    subprocess.run([os.path.join(ORCA_DIR, 'orca_pltvib'), name, str(mode)], cwd=folder, capture_output=True)
    frames = open(f'{out}.v{mode:03d}.xyz').read()
    view = py3Dmol.view(width=width, height=height)
    view.addModelsAsFrames(frames, 'xyz')
    view.setStyle({'stick': {'radius': 0.15}, 'sphere': {'scale': 0.25}})
    view.animate({'loop': 'backAndForth'})
    view.zoomTo()
    return view.show()

show_mode(out, 6)     # try also 7 and 8

Finally, the frontier orbitals: the HOMO (highest occupied molecular orbital, number 4: water has 10 electrons in 5 doubly occupied orbitals numbered 0–4) and the LUMO (lowest unoccupied orbital, number 5).

In [ ]:
show_cube('water/water_r2SCAN-3c/water.homo.cube', isovalue=0.05, signed=True)
show_cube('water/water_r2SCAN-3c/water.lumo.cube', isovalue=0.05, signed=True)

### Atomic charges

Atomic charges are not observables: there are many ways of dividing the electron density among the atoms, and they give different numbers. Compare them with the charges of the water models used in MD (TIP3P: O −0.834, H +0.417; SPC/E: O −0.8476, H +0.4238).

In [ ]:
for kind in ['MULLIKEN', 'LOEWDIN', 'HIRSHFELD', 'CHELPG']:
    print(f'{kind:10s}', '  '.join(f'{el} {q:+.3f}' for el, q in charges(out, kind)))

### How good is the method?

The choice of the method and of the basis set is always a compromise between accuracy and cost. Here we optimise water with five combinations and record geometry, bending frequency, dipole and computer time. HF is the cheapest, B3LYP the most used hybrid functional (with the D4 dispersion correction), and MP2 (second-order perturbation theory on top of HF) the cheapest *wavefunction* method including correlation. All together, about 2 minutes.

In [ ]:
ladder = ['HF def2-SVP', 'HF def2-TZVP', 'B3LYP D4 def2-SVP', 'B3LYP D4 def2-TZVP', 'r2SCAN-3c', 'RI-MP2 def2-TZVP def2-TZVP/C']
table = []
for method in ladder:
    name = 'water_' + method.split()[0] + '_' + method.split()[-1].replace('/', '')
    if method == 'r2SCAN-3c':
        name = 'water_r2SCAN-3c_noprop'
    out = run_orca(name, f'! {method} Opt Freq TightSCF\n' + water, folder='water')
    r1, r2, angle = geometry(os.path.join('water', name, name + '.xyz'))
    table.append((method, r1, angle, frequencies(out)[:, 0].min(), dipole(out), wall_time(out)))
print(f"{'method':30s} {'O-H (Å)':>8s} {'angle':>7s} {'bend (cm-1)':>12s} {'dipole (D)':>11s} {'time (s)':>9s}")
for row in table:
    print('{:30s} {:8.4f} {:7.2f} {:12.0f} {:11.2f} {:9.1f}'.format(*row))
print('{:30s} {:8.4f} {:7.2f} {:12.0f} {:11.2f}'.format('experiment (harmonic)', 0.9572, 104.52, 1649, 1.85))

### Two molecules: the hydrogen bond

The energy of an interaction between two molecules A and B is computed as a difference:

$$\Delta E = E(AB) - E(A) - E(B)$$

where all three are computed at the geometry they have in the complex. The numbers are small differences of large numbers, so it is a good idea to compute them in kcal/mol.

There is a subtle problem. In the complex, molecule A can use also the basis functions of molecule B to improve its own description, and this lowers $E(AB)$ artificially: the interaction looks stronger than it is. This is the **basis-set superposition error** (BSSE). The standard fix is the **counterpoise correction**: $E(A)$ and $E(B)$ are computed in the basis set of the whole complex, putting *ghost atoms* (basis functions without nuclei and electrons) at the positions of the atoms of the partner. In ORCA a ghost atom is written with a colon after the element symbol, e.g. `H:`.

The geometry of the water dimer is taken from the S22 database, a set of 22 complexes with very accurate reference interaction energies (CCSD(T) extrapolated to the complete basis set, CBS). The reference value for the water dimer is −5.02 kcal/mol. The folder `Data/qm/s22` contains all the S22 complexes: **GROUP 1** uses them in the report.

In [ ]:
import pandas as pd
s22 = pd.read_csv(os.path.join(QM_DIR, 's22', 's22x5_reference.csv'))
s22[s22.scale == 1.0][['id', 'system', 'nA', 'nB', 'Eint_ref_kcal']]

In [ ]:
def interaction_energy(xyzfile, nA, method, folder, counterpoise=True, q=(0, 0, 0)):
    # interaction energy (kcal/mol) of a dimer whose first nA atoms are molecule A and the others molecule B
    # q = (charge of AB, charge of A, charge of B); all species are closed-shell singlets
    nat = int(open(xyzfile).readline())
    tag = os.path.basename(xyzfile).replace('.xyz', '') + '_' + method.replace(' ', '_').replace('/', '')
    A, B = range(nA), range(nA, nat)
    jobs = {'AB': (xyz_block(xyzfile), q[0])}
    if counterpoise:                                     # monomers in the basis of the dimer: the partner is made of ghost atoms
        jobs['A'] = (xyz_block(xyzfile, ghost=B), q[1])
        jobs['B'] = (xyz_block(xyzfile, ghost=A), q[2])
    else:                                                # monomers in their own basis: the partner is removed
        lines = xyz_block(xyzfile).splitlines()
        jobs['A'] = ('\n'.join(lines[:nA]), q[1])
        jobs['B'] = ('\n'.join(lines[nA:]), q[2])
    E = {}
    for k, (coords, charge) in jobs.items():
        name = f'{tag}_{k}' + ('_cp' if counterpoise and k != 'AB' else '')
        out = run_orca(name, f'! {method} TightSCF\n* xyz {charge} 1\n{coords}\n*', folder=folder)
        E[k] = final_energy(out)
    return (E['AB'] - E['A'] - E['B']) * HARTREE2KCAL

In [ ]:
xyz = os.path.join(QM_DIR, 's22', '02_Water_dimer_1.0.xyz')
show_molecule(xyz)
os.makedirs('water_dimer', exist_ok=True)
for method in ['HF def2-SVP', 'HF def2-TZVP', 'B3LYP def2-SVP', 'B3LYP def2-TZVP', 'B3LYP D4 def2-TZVP']:
    e_nocp = interaction_energy(xyz, 3, method, 'water_dimer', counterpoise=False)
    e_cp = interaction_energy(xyz, 3, method, 'water_dimer', counterpoise=True)
    print(f'{method:20s}  no CP: {e_nocp:6.2f}   CP: {e_cp:6.2f}   BSSE: {e_nocp - e_cp:5.2f} kcal/mol')
print('reference CCSD(T)/CBS: -5.02 kcal/mol')

The BSSE is large with the small basis set and becomes smaller with the larger one. Notice also that the interaction energy of the water dimer is already well described by HF: the hydrogen bond is mostly electrostatic. This is not the case for all interactions, as GROUP 1 will discover.

## Part 5: light, excited states with TD-DFT

A molecule absorbs a photon when its energy $E = hc/\lambda$ matches the energy difference between the ground state and an excited electronic state. A simple estimate of the lowest excitation is the HOMO–LUMO gap, but the excited state is not just "one electron moved to the LUMO": the other electrons respond. **Time-dependent DFT** (TD-DFT) computes the excitation energies and, for each of them, the **oscillator strength** $f$, a dimensionless number proportional to the intensity of the absorption band. States with $f \approx 0$ are *dark*: they exist but do not absorb light.

Useful conversions: $\lambda$ (nm) = 1239.84 / $E$ (eV); visible light goes from about 400 nm (violet, 3.1 eV) to 700 nm (red, 1.8 eV). A molecule looks of the complementary colour of the light it absorbs.

We start from formaldehyde, H$_2$C=O: we optimise it and compute the first 6 singlet excited states with the range-separated hybrid functional CAM-B3LYP and a basis set with diffuse functions (def2-SVPD), useful for excited states.

In [ ]:
os.makedirs('tddft', exist_ok=True)
formaldehyde = '''* xyz 0 1
C   0.000   0.000   0.000
O   0.000   0.000   1.210
H   0.000   0.940  -0.590
H   0.000  -0.940  -0.590
*'''
opt = run_orca('formaldehyde_opt', '! r2SCAN-3c Opt TightSCF\n' + formaldehyde, folder='tddft')
xyz = 'tddft/formaldehyde_opt/formaldehyde_opt.xyz'
inp = '! CAM-B3LYP def2-SVPD TightSCF\n%tddft\n  NRoots 6\nend\n' + \
      plots_block(['MO("formaldehyde.homo.cube",7,0)', 'MO("formaldehyde.lumo.cube",8,0)']) + \
      '* xyz 0 1\n' + xyz_block(xyz) + '\n*'
out = run_orca('formaldehyde_tddft', inp, folder='tddft')

In [ ]:
def excitations(out):
    # excited states from the absorption spectrum table: list of (energy eV, wavelength nm, oscillator strength)
    text = open(out).read().split('ABSORPTION SPECTRUM VIA TRANSITION ELECTRIC DIPOLE MOMENTS')[1].split('\n\n')[0]
    rows = []
    for l in text.splitlines():
        if '->' in l:
            f = l.split('->')[1].split()
            rows.append((float(f[1]), float(f[3]), float(f[4])))
    return np.array(rows)

def transitions(out):
    # the orbital contributions to each excited state, as printed by ORCA (e.g. "7a -> 8a : 0.98")
    text = open(out).read().split('EXCITED STATES')[1].split('ABSORPTION SPECTRUM')[0]
    for block in re.split(r'\n(?=STATE\s+\d+:)', text)[1:]:
        head = block.splitlines()[0]
        contr = re.findall(r'(\d+a\s+->\s+\d+a)\s+:\s+(\d\.\d+)', block)
        print(head.split('<S**2>')[0].strip(), '  ', ',  '.join(f'{t} ({c})' for t, c in contr))

def spectrum(states, fwhm=0.3, emin=1.5, emax=8.0):
    # absorption spectrum: a Gaussian of width fwhm (eV) for each state, with area proportional to f
    E = np.linspace(emin, emax, 1000)
    sigma = fwhm / 2.3548
    eps = sum(f * np.exp(-(E - e)**2 / (2 * sigma**2)) / (sigma * np.sqrt(2 * np.pi)) for e, lam, f in states)
    return E, eps

In [ ]:
states = excitations(out)
print(' E (eV)  lambda (nm)  f')
print(np.round(states, 4))
transitions(out)

The first excited state is the $n \rightarrow \pi^*$ transition: an electron moves from the lone pair of the oxygen (the HOMO) to the antibonding $\pi^*$ orbital of the C=O bond (the LUMO). It is almost dark ($f \approx 0$): the two orbitals have different symmetry, and the transition is "forbidden". The experimental vertical excitation energy is about 4.0 eV (≈ 310 nm, UV light): formaldehyde, like all simple carbonyls, is colourless.

In [ ]:
show_cube('tddft/formaldehyde_tddft/formaldehyde.homo.cube', isovalue=0.05, signed=True)
show_cube('tddft/formaldehyde_tddft/formaldehyde.lumo.cube', isovalue=0.05, signed=True)

### Conjugation shifts the absorption to longer wavelengths

Now *trans*-butadiene, CH$_2$=CH–CH=CH$_2$: two conjugated double bonds. The bright $\pi \rightarrow \pi^*$ transition (experiment: 5.9 eV, 210 nm) has a large oscillator strength. In the report, GROUP 2 will see what happens with longer conjugated chains, like the one of retinal, the chromophore of vision.

In [ ]:
butadiene = '''* xyz 0 1
C   -1.80   0.42   0.0
C   -0.60  -0.20   0.0
C    0.60   0.20   0.0
C    1.80  -0.42   0.0
H   -2.72  -0.16   0.0
H   -1.88   1.51   0.0
H   -0.55  -1.29   0.0
H    0.55   1.29   0.0
H    2.72   0.16   0.0
H    1.88  -1.51   0.0
*'''
opt = run_orca('butadiene_opt', '! r2SCAN-3c Opt TightSCF\n' + butadiene, folder='tddft')
xyz = 'tddft/butadiene_opt/butadiene_opt.xyz'
out = run_orca('butadiene_tddft', '! CAM-B3LYP def2-SVPD TightSCF\n%tddft\n  NRoots 6\nend\n* xyz 0 1\n' + xyz_block(xyz) + '\n*', folder='tddft')
transitions(out)
fig, ax = plt.subplots(figsize=(6, 3.5))
for name, label in [('formaldehyde_tddft', 'formaldehyde'), ('butadiene_tddft', 'butadiene')]:
    st = excitations(f'tddft/{name}/{name}.out')
    E, eps = spectrum(st)
    ax.plot(E, eps, label=label)
    ax.vlines(st[:, 0], 0, st[:, 2] * 3, colors=ax.lines[-1].get_color())
ax.set_xlabel('excitation energy (eV)'); ax.set_ylabel('absorption (arb. units)'); ax.legend()
secax = ax.secondary_xaxis('top', functions=(lambda e: 1239.84 / np.maximum(e, 1e-3), lambda l: 1239.84 / np.maximum(l, 1e-3)))
secax.set_xlabel('wavelength (nm)')
plt.tight_layout(); plt.show()

## What next

You are now ready for the report. The training questions are answered by everybody; then **GROUP 1** goes on with the interaction energies of the S22 complexes (Part 4) and **GROUP 2** with the excited states of biological chromophores (Part 5). The report notebook contains the same setup and helper functions, so the calculations of the training (in your `T07` folder on Google Drive) are reused there.